In [0]:
# Nordstream 2022-09-26
https://simple.wikipedia.org/wiki/2022_Nord_Stream_pipeline_sabotage
latitude 54.868333, longitude 15.401667 --  54°52.6′N 15°24.6′E[1]
latitude 55.533611, longitude 15.685833 --  55°32.1′N 15°41.9′E[2]
latitude 55.551111, longitude 15.784167  --    55°33.4′N 15°47.3′E[2]
latitude 55.545833 , longitude 15.768611  --  55°32.45′N 15°46.74′E[2]

https://geopandas.org/en/stable/gallery/polygon_plotting_with_folium.html

In [0]:
%sql
/* # Nordstream 2022-09-26
https://simple.wikipedia.org/wiki/2022_Nord_Stream_pipeline_sabotage 
5 * 1.852km
*/ 
SELECT t.area, t.latitude, t.longitude
FROM ( VALUES  ('Danger area 1', 54.876667, 15.41)
             , ('Danger area 2', 55.535, 15.698333 )
              , ('Danger area 3', 55.556667, 15.788333 )
              , ('Danger area 4', 55.540833, 15.779 )
      ) t(area, latitude, longitude)

In [0]:
ST_DistanceSphere( ST_Point(long1, lat1), ST_Point(long2, lat2)) < 5*1852 -- 5 nautic miles in meters

In [0]:
%sql
CREATE WIDGET TEXT schema_name DEFAULT "vessel";

In [0]:
%sql
SELECT *, ST_GeomFromWKT(geometry)
 FROM ${schema_name}.sea_zones
WHERE zone IN ('Er5')

In [0]:
%sh
pip install geopandas geodatasets folium

In [0]:
%python
import pandas as pd
import geopandas as gpd
import geodatasets
import folium
import matplotlib.pyplot as plt

In [0]:
%sql
CREATE OR REPLACE zones
as
SELECT *, ST_GeomFromWKT(geometry)
 FROM ${schema_name}.sea_zones
WHERE zone IN ('Er5')
;
/* combine with other circles */

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW points
as
/* to do: limit on mmsi, minute level 
   latitude between 54.0 and 56.0
   longitude between 14.0 and 16.0
*/
SELECT v.* /*, ST_AsText(ST_Point(v.longitude, v.latitude)) point_geom*/
FROM vessel.messages v 
WHERE v.yyyy_mm_dd = '2022-09-26'
and exists (SELECT *, ST_GeomFromWKT(geometry)
            FROM vessel.sea_zones z
            WHERE z.zone IN ('Er5')
            and ST_Contains(
                            ST_GeomFromText(z.geometry),  -- your WKT geometry
                            ST_Point(v.longitude, v.latitude)       -- your point
       )
            )

In [0]:
df = spark.sql("SELECT * FROM zones WHERE zone IN ('Er5')").toPandas() # Convert SQL result to Pandas DataFrame
zones_gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['geometry']))

In [0]:
points_df = spark.sql("SELECT * FROM points ").toPandas() # Convert SQL result to Pandas DataFrame
#points_gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['point_geom']))
#points_gdf.head()

In [0]:
zones_gdf.plot(figsize=(6, 6))
plt.show()

In [0]:
m = folium.Map(location=[55.1, 15.3], zoom_start=10, tiles="CartoDB positron")
m

In [0]:
for _, r in zones_gdf.iterrows():
    sim_geo = gpd.GeoSeries(r["geometry"]).simplify(tolerance=0.001) # Simplify the geometry
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fillColor": "orange"})
    folium.Popup(r["zone_title_prefix_en"]).add_to(geo_j)
    geo_j.add_to(m)
m

In [0]:

for _, r in points_gdf.iterrows():
    lat = r["latitude"]
    lon = r["longitude"]    
    folium.Marker(
        location=[lat, lon],
        popup=f"""name: {r["name"]} <br> type: {r["ship_type"]}""",
    ).add_to(m)

m